### Structure Track: Workflow to Submit Structure Predictions on the OpenADMET CYP Blind Challenge
### focus is on using OpenFold3 as a co-folding tool
### output from this is put through a variety of selection methods (md, knowledge based, etc)
### Several cells borrowed outright or adapted from the excellent tutorial provided by OpenADMET
### Otherwise - created by Byron DeLaBarre / The Consulting Biochemist Sept 2026
### Claude Opus 5.5/5.0 was used to generate first pass code that was modified where necessary

In [ ]:
# Note - use 'PoseBusters' environment

import sys
import warnings

from pathlib import Path
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import time, requests

from tqdm.auto import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem, DataStructs

from matplotlib.offsetbox import OffsetImage, AnnotationBbox

from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.inchi import MolToInchiKey

from pymol import cmd

import json
import shutil

import biotite.structure.io as strucio

import os, glob
import gemmi
from posebusters import PoseBusters

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

from gradio_client import Client, handle_file
from huggingface_hub import get_token

from datetime import datetime, timezone
from datetime import datetime, timezone, timedelta

tqdm.pandas()
sns.set_style("whitegrid")
sns.set_context("notebook")

def rewrite_ligand_resname_pdb(
    input_structure: Path,
    output_pdb: Path,
    old_resname: str = "LIG1",
    new_resname: str = "LIG",
):
    """Read a structure file and write a PDB with ligand residue name normalized."""
    if strucio is None:
        raise ImportError("biotite is required for residue renaming. Install with: pip install biotite")

    # Generic load handles CIF/PDB based on extension
    stack = strucio.load_structure(str(input_structure))

    if hasattr(stack, "res_name"):
        mask = stack.res_name == old_resname
        if np.any(mask):
            stack.res_name[mask] = new_resname

    strucio.save_structure(str(output_pdb), stack)
    return output_pdb

In [ ]:
load_dotenv(find_dotenv(), override=True)
#print(os.getenv("CHALLENGE_CSV"))
test_structure = pd.read_csv(Path(os.getenv("CHALLENGE_CSV")))

required_cols = ["id", "SMILES"]
missing_cols = [col for col in required_cols if col not in test_structure.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in test_structure: {missing_cols}")

structure_df = test_structure[required_cols].copy().rename(columns={"Molecule_Name": "id"})
structure_df["mol"] = structure_df["SMILES"].apply(Chem.MolFromSmiles)

invalid_smiles = structure_df["mol"].isna().sum()
print(f"Total ligands: {len(structure_df)}")
print(f"Invalid SMILES: {invalid_smiles}")
structure_df.head()

## 2. Preparing the data for prediction

For the structure track, the main input is the ligand SMILES list so preprocessing is minimal:

- Keep molecule identifiers and SMILES.
- Parse SMILES into RDKit molecules.
- Define the protein context for the structure-generation pipeline.

For this tutorial, we load the CYP3A4 protein sequence from a local FASTA file stored in `inputs/CYP3A4_protein_sequence.fasta`.

You will be predicting 20 structures in total.

## 6. Prepare submission and validate

The following are the submission requirements for this track:

- Upload a `.zip` archive containing one PDB file per ligand in the blinded structure set.
- Name each file `{id}.pdb` (e.g., matching the `id` column in `structure_df`).
- In each file, the ligand residue name must be exactly **`LIG`**.



In [ ]:
# --- CONFIGURATION ---

load_dotenv(find_dotenv(), override=True)


RUN_NAME = os.getenv("RUN_NAME")
HARVEST_INPUT_DIR = Path(os.getenv("CYP_PREDICTIONS")) / RUN_NAME
OUTPUT_DIR  = HARVEST_INPUT_DIR / "harvested_top"          # best pose per compound
ALL_DIR     = HARVEST_INPUT_DIR / "all_predictions"    # every pose, flat, for visual inspection
SUMMARY_CSV = HARVEST_INPUT_DIR / f"harvest_summary_{RUN_NAME}.csv"
ALL_SCORES_CSV = HARVEST_INPUT_DIR / f"harvest_all_scores_{RUN_NAME}.csv"   # one row per pose
SKIP_DIRS   = {"logs", OUTPUT_DIR.name, ALL_DIR.name}   # <-- don't scan our own outputs

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_DIR.mkdir(parents=True, exist_ok=True)

 We use this helper to convert our  predictions  into submission-ready PDBs. Adjust `cif_dir` to point to your own prediction directory here.
 

In [ ]:
# PoseBusters analysis for predicted CYP3A4 structures
# (optional - OpenADMET will run on submission and failed structures won't be scored)
#
# Fixed 23 Oct 2026:
#   - bool(pd.NA) raises, which full_report=True produces -> every compound
#     errored and nothing was evaluated
#   - the conditioning file kept only chain A, removing the heme, so
#     minimum_distance_to_organic_cofactors had no cofactor to measure
#   - Chem.PathToSubmol takes BOND indices, not atom indices
#   - the template carried explicit H while the submission ligand has none
#   - the fallback set every bond to SINGLE, guaranteeing chemistry failures
#   - the summary inferred boolean columns with isin([True, False]), which
#     admits numeric columns whose values are 0/1 (pandas: 0 == False), so
#     measured counts of zero were reported as failed checks

RDLogger.DisableLog("rdApp.*")
load_dotenv(find_dotenv(), override=True)

# ---- config ----
SUBMISSION_DIR = Path(os.getenv("best_pdb_dir"))
CHALLENGE_CSV = os.getenv("CHALLENGE_CSV")  # id, smiles
EXTRACT_DIR = os.getenv("EXTRACT_DIR")
OUT_CSV = os.getenv("OUT_CSV")
ID_COL, SMILES_COL = "id", "SMILES"
PROTEIN_CHAIN  = "A"
LIGAND_RESNAME = "LIG"      # after your LIG0 -> LIG rename
HEME_RESNAMES  = ("HEM", "HEC", "HEB")

# the checks OpenADMET actually zeroed scores on
PB_CRITICAL = ["minimum_distance_to_organic_cofactors", "bond_angles",
               "internal_steric_clash", "bond_lengths", "sanitization",
               "all_atoms_connected", "double_bond_stereochemistry",
               "tetrahedral_chirality", "minimum_distance_to_protein",
               "minimum_distance_to_inorganic_cofactors",
               "volume_overlap_with_protein",
               "volume_overlap_with_organic_cofactors"]
# ----------------
print(f"EXTRACT_DIR: {EXTRACT_DIR}")
print(f"SUBMISSION_DIR: {SUBMISSION_DIR}")
print(f"CHALLENGE_CSV: {CHALLENGE_CSV}")
print(f"OUT_CSV: {OUT_CSV}")
os.makedirs(EXTRACT_DIR, exist_ok=True)

# --- reference 2D mols from challenge SMILES, keyed by submission filename stem ---
# No AddHs: the submission ligand has its hydrogens stripped, and a template
# carrying explicit H will not substructure-match it.
df_master = pd.read_csv(CHALLENGE_CSV)
lm_dict = {}
for _, row in df_master.iterrows():
    smi = row[SMILES_COL]
    if pd.notna(smi):
        m = Chem.MolFromSmiles(str(smi))
        if m:
            key = str(row[ID_COL]).strip().removeprefix("cmpd_").upper()
            lm_dict[key] = m
print(f"{len(lm_dict)} reference molecules: {sorted(lm_dict)[:5]} ...")


def lig_heme_contact(pdb_path):
    """Minimum heavy-atom distance between the LIG residue and the heme.

    PoseBusters reports only a boolean for the cofactor check, so measure it
    directly to see how far outside the threshold a failing pose sits.
    -> (distance_A, ligand_atom, heme_atom) or (nan, None, None)
    """
    st = gemmi.read_structure(str(pdb_path)); st.setup_entities()
    L, H = [], []
    for ch in st[0]:
        for r in ch:
            nm = r.name.strip().upper()
            tgt = L if nm == LIGAND_RESNAME else (H if nm in HEME_RESNAMES else None)
            if tgt is None:
                continue
            for a in r:
                if a.element.name != "H":
                    tgt.append((a.name.strip(), np.array(a.pos.tolist())))
    if not L or not H:
        return np.nan, None, None
    best = (np.inf, None, None)
    for ln, lp in L:
        for hn, hp in H:
            d = float(np.linalg.norm(lp - hp))
            if d < best[0]:
                best = (d, ln, hn)
    return best


pb = PoseBusters(config="dock")
pdb_files = sorted(glob.glob(os.path.join(SUBMISSION_DIR, "*.pdb")))
print(f"{len(pdb_files)} submission PDBs")

pb_results = []
bool_checks = set()          # column names that were genuinely boolean
for pdb_path in tqdm(pdb_files, desc="PoseBusters"):
    clean_id = Path(pdb_path).stem
    if clean_id not in lm_dict:
        print(f"  [!] no reference SMILES for {clean_id}; skipping")
        continue
    ref_mol = lm_dict[clean_id]
    protein_pdb = os.path.join(EXTRACT_DIR, f"{clean_id}_protein.pdb")

    try:
        # 1. conditioning file: protein + heme, ONLY the ligand removed.
        # The heme must stay or minimum_distance_to_organic_cofactors has
        # nothing to measure against.
        st = gemmi.read_structure(pdb_path)
        prot = st.clone()
        for model in prot:
            drop = [c.name for c in model
                    if any(r.name.strip() == LIGAND_RESNAME for r in c)]
            for cname in drop:
                model.remove_chain(cname)
        prot.write_pdb(protein_pdb)

        # 2. extract the compound by RESNAME (not chain) so heme can't be
        # picked up. SplitMolByPDBResidues keys on residue name;
        # PathToSubmol would need bond indices, not atom indices.
        mol_complex = Chem.MolFromPDBFile(pdb_path, sanitize=False, removeHs=False)
        frags = Chem.SplitMolByPDBResidues(mol_complex)
        pred_raw = frags.get(LIGAND_RESNAME)
        if pred_raw is None or pred_raw.GetNumAtoms() == 0:
            raise ValueError(f"no residue named {LIGAND_RESNAME} found "
                             f"(residues present: {sorted(frags)})")

        # 3. graft correct bond orders from the SMILES template onto the 3D
        # coords. No fallback: forcing every bond to SINGLE guarantees the
        # chemistry checks fail and conceals the real cause.
        pred_fixed = AllChem.AssignBondOrdersFromTemplate(ref_mol, pred_raw)
        Chem.SanitizeMol(pred_fixed)

        # 4. bust
        result = pb.bust(mol_pred=pred_fixed, mol_true=None,
                         mol_cond=protein_pdb, full_report=True)

        # 5. record. pd.isna is tested BEFORE any boolean coercion, and only
        # genuine bools are treated as checks -- a measured count of 0 is not
        # a failure.
        ignore = ["mol_pred", "mol_true", "mol_cond", "mol_true_loaded"]
        test_cols = [c for c in result.columns if c not in ignore]
        row = {"compound": clean_id, "pb_error": None}
        n_pass = n_total = 0
        for col in test_cols:
            v = result[col].iloc[0]
            key = f"pb_{col.replace('pb_', '')}"
            if pd.isna(v):
                row[key] = None
            elif isinstance(v, (bool, np.bool_)):
                bool_checks.add(key)
                row[key] = bool(v)
                n_pass += int(v)
                n_total += 1
            else:
                row[key] = v
        row.update(pb_pass=(n_total > 0 and n_pass == n_total),
                   pb_n_pass=n_pass, pb_n_total=n_total)

        d, la, ha = lig_heme_contact(pdb_path)
        row.update(lig_heme_min_dist=d, lig_heme_lig_atom=la,
                   lig_heme_heme_atom=ha)
        pb_results.append(row)

    except Exception as e:
        pb_results.append({"compound": clean_id, "pb_pass": False,
                           "pb_n_pass": 0, "pb_n_total": 0, "pb_error": str(e)})

pb_df = pd.DataFrame(pb_results)
pb_df.to_csv(OUT_CSV, index=False)

bool_cols = sorted(c for c in bool_checks if c in pb_df.columns)
crit_cols = [f"pb_{c}" for c in PB_CRITICAL if f"pb_{c}" in pb_df.columns]

print(f"\n=== PoseBusters Summary ===")
print(f"Total:             {len(pb_df)}")
print(f"checks evaluated:  {pb_df['pb_n_total'].min()}-"
      f"{pb_df['pb_n_total'].max()} per compound "
      f"({len(bool_cols)} boolean columns)")
print(f"All 31 checks pass: {pb_df['pb_pass'].sum()}   "
      f"(expected to be 0 -- most_extreme_clash_* and not_too_far_away_waters "
      f"fail universally and OpenADMET does not enforce them; read the "
      f"Critical line instead)")
if crit_cols:
    crit_ok = pb_df[crit_cols].apply(lambda r: all(v is not False for v in r),
                                     axis=1)
    print(f"Critical pass:     {int(crit_ok.sum())}/{len(pb_df)}   "
          f"({len(crit_cols)} of {len(PB_CRITICAL)} critical checks present)")
if pb_df["pb_error"].notna().any():
    print("\nerrors:")
    for _, r in pb_df[pb_df["pb_error"].notna()].iterrows():
        print(f"  {r['compound']}: {r['pb_error']}")

fails = pb_df[bool_cols].apply(lambda c: (c == False).sum())
fails = fails[fails > 0].sort_values(ascending=False)
print(f"\nFailures by check (of {len(pb_df)}):"
      f"\n{fails.to_string() if len(fails) else '  none'}")

print("\nper-compound failures (critical checks marked *):")
any_fail = False
for _, r in pb_df.iterrows():
    bad = [("*" if c in crit_cols else "") + c.replace("pb_", "")
           for c in bool_cols if r.get(c) is False]
    if bad:
        any_fail = True
        print(f"  {r['compound']}: {bad}")
if not any_fail:
    print("  none")

print("\nmeasured ligand-heme minimum heavy-atom distance:")
cof = "pb_minimum_distance_to_organic_cofactors"
show = ["compound", "lig_heme_min_dist", "lig_heme_lig_atom",
        "lig_heme_heme_atom"] + ([cof] if cof in pb_df.columns else [])
print(pb_df[show].sort_values("lig_heme_min_dist").round(2).to_string(index=False))

print(f"\n-> {OUT_CSV}")

In [ ]:
# strip all hydrogens for submission (could move this to the refine/rank step - leaving it here for now.  It is a good final check before doing he last QC steps)
from pathlib import Path
import gemmi
load_dotenv(find_dotenv(), override=True)
SRC_DIR = Path(os.getenv("SUBMISSION_DIR"))       # <-- set this
print(f"SRC_DIR: {SRC_DIR}")
# if not SRC_DIR.is_dir():
#     raise FileNotFoundError(f"SUBMISSION_DIR does not exist: {SRC_DIR}. "
#                             f"Check the SUBMISSION_DIR value in your .env file.")
OUT_DIR = SRC_DIR / "no_h"                      # originals left untouched
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUT_DIR: {OUT_DIR}")
rows = []
for p in sorted(SRC_DIR.glob("*.pdb")):
    st = gemmi.read_structure(str(p)); st.setup_entities()
    n0 = sum(len(r) for ch in st[0] for r in ch)
    nres0, nch0 = sum(len(ch) for ch in st[0]), len(st[0])

    st.remove_hydrogens()
    st.setup_entities()
    out = OUT_DIR / p.name
    st.write_pdb(str(out))

    chk = gemmi.read_structure(str(out)); chk.setup_entities()
    n1 = sum(len(r) for ch in chk[0] for r in ch)
    nH = sum(1 for ch in chk[0] for r in ch for a in r if a.element.name == "H")
    rows.append({"file": p.name, "atoms_before": n0, "atoms_after": n1,
                 "H_removed": n0 - n1, "H_left": nH,
                 "chains": f"{nch0}->{len(chk[0])}",
                 "residues": f"{nres0}->{sum(len(ch) for ch in chk[0])}",
                 "chain_ids": "".join(ch.name for ch in chk[0])})

import pandas as pd
R = pd.DataFrame(rows)
print(R.to_string(index=False))
print(f"\n{len(R)} files -> {OUT_DIR}")
assert (R.H_left == 0).all(), "hydrogens survived in some files"
print("no hydrogens remain; residue and chain counts unchanged where "
      f"'->' shows equal values")

In [ ]:
# PDB files  in `output_dir` (one `{id}.pdb` per ligand)

# This OpenADMET validation code also checks that your submitted structures have the correct bond orders 
# (corresponding to the SMILES given) and that each complex has at most three chains — protein, heme cofactor, and ligand. 
# The validation function takes an `id: SMILES` mapping, which we build directly from `structure_df`.



load_dotenv(find_dotenv(), override=True)

# Find the project root before importing the validation module
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

#from validation.structure_validation import validate_structure_submission
from validation import validate_structure_submission

# Point this to your directory of predicted PDB files

processed_submission_dir = Path(os.getenv("best_pdb_dir"))
zip_path = Path(os.getenv("zip_path"))



# Build an ID -> SMILES mapping for ligand chemistry validation (connectivity and bond orders).
expected_ligand_smiles = dict(zip(structure_df["id"], structure_df["SMILES"]))


pdb_files = sorted(processed_submission_dir.glob("*.pdb"))
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for pdb in pdb_files:
        zf.write(pdb, arcname=pdb.name)
print(f"Processing submission directory: {processed_submission_dir}")
print(f"Created zip archive: {zip_path} ({len(pdb_files)} files)")
print(f"Built SMILES map for {len(expected_ligand_smiles)} ligands")

# Suppress noisy warning logs from RDKit and MDAnalysis during validation.
RDLogger.DisableLog("rdApp.warning")
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module=r"MDAnalysis.*")
        warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"MDAnalysis.*")

        is_valid, validation_errors = validate_structure_submission(
            zip_path,
            expected_ids=set(structure_df["id"]),
            expected_ligand_smiles=expected_ligand_smiles,
            require_lig_resname=True,
            check_numbering=True,
        )
finally:
    RDLogger.EnableLog("rdApp.warning")

if is_valid:
    print("✅ Submission is valid — ready to upload.")
else:
    print("❌ Submission is invalid:")
    for msg in validation_errors:
        print(f"  - {msg}")

In [ ]:
# Double check the zip path and file before the API submission!
load_dotenv(find_dotenv(), override=True)
print(os.getenv("zip_path"))

## Submission via API - taken directly from tutorial

Go to the web portal and submit there, carefully filling in your information.

https://huggingface.co/spaces/openadmet/cyp-challenge

In [ ]:
from datetime import datetime, timezone


load_dotenv(find_dotenv(), override=True)
# get a token to submit
hf_token = get_token()
client = Client("openadmet/cyp-challenge", token=hf_token)

# Replace this with the path to your submission zip file
#file_path
# Replace this with your own information and uncomment to submit.
result = client.predict(
   username= os.getenv("username"),
   user_alias=os.getenv("user_alias"), # Replace if anonymous
   anon_checkbox=True, # Change to True if you want to submit anonymously
   participant_name=os.getenv("participant_name"),
   discord_username=os.getenv("discord_username"),
   email=os.getenv("email"),
   affiliation=os.getenv("affiliation"),
   model_tag=os.getenv("model_tag"),
   paper_checkbox=True,
   proprietary_data_checkbox=False,
   open_code_checkbox=True,
   track_select="Structure Prediction",
   file_input=handle_file(os.getenv("zip_path")),
   api_name="/submit_predictions",
)

print("Last run:", datetime.fromtimestamp(time.time(), tz=timezone(timedelta(hours=-4))).strftime("%Y-%m-%d %H:%M:%S EDT"))
print(result)